# Fast Radio Burst Detection

### download data

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# For time series
from typing import List
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import userdata

In [2]:
def download_kaggle(kaggle_command="kaggle competitions download -c liver-fibrosis-severity-prediction"):

  # Get Kaggle Key
  kaggle_username = userdata.get("KAGGLE_USER")
  kaggle_key = userdata.get("KAGGLE_KEY")
  if not kaggle_username or not kaggle_key:
      print("Error: Kaggle_USERNAME or Kaggle_KEY not found in Colab Secrets.")
      return

  # Write the credentials to ~/.kaggle/kaggle.json
  kaggle_dir = os.path.expanduser("~/.kaggle")
  os.makedirs(kaggle_dir, exist_ok=True)

  # Create JSON
  kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
  with open(kaggle_json_path, "w") as f:
      f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
  os.chmod(kaggle_json_path, 0o600)

  try:
      os.system(kaggle_command)
      print("\n--- Download complete! ---")
      os.system("ls -la")
      os.system("unzip -o '*.zip' && rm -f *.zip")
      os.system("ls -la")

  except Exception as e:
      print(f"An error occurred during download: {e}")

In [3]:
download_kaggle(
    "kaggle competitions download -c individual-test-fast-radio-burst-detection"
)


--- Download complete! ---


In [5]:
# !mkdir /content/train/npy /content/train/label
# !mv /content/train/train/*.npy /content/train/npy
# !mv /content/train-labels-corrected/train/*.csv /content/train/label
# !rm -rf train/train/

mkdir: cannot create directory ‘/content/train/npy’: File exists
mkdir: cannot create directory ‘/content/train/label’: File exists
mv: cannot stat '/content/train/train/*.npy': No such file or directory
mv: cannot stat '/content/train-labels-corrected/train/*.csv': No such file or directory


# function

In [15]:
import numpy as np
import pandas as pd
import os
import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

# Import optional libraries
try:
    import xgboost as xgb
    HAS_XGBOOST = True
    print("✓ XGBoost available")
except ImportError:
    HAS_XGBOOST = False
    print("✗ XGBoost not available")

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
    print("✓ LightGBM available")
except ImportError:
    HAS_LIGHTGBM = False
    print("✗ LightGBM not available")

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
    print("✓ CatBoost available")
except ImportError:
    HAS_CATBOOST = False
    print("✗ CatBoost not available")

from scipy import signal, stats
from scipy.fft import fft, fftfreq
import pywt
from skimage import measure
import warnings
warnings.filterwarnings('ignore')

# Signal type constants
SIGNAL_TYPES = ['pulse', 'broad', 'narrow']
print(f"\nSignal types: {SIGNAL_TYPES}")


✓ XGBoost available
✓ LightGBM available
✗ CatBoost not available

Signal types: ['pulse', 'broad', 'narrow']


In [16]:
def parse_label(label_str):
    """Convert label string to multi-label format
    Returns: [pulse, broad, narrow] where 1 indicates presence of signal type
    """
    label_vec = [0, 0, 0]

    if label_str == 'None' or pd.isna(label_str):
        return label_vec

    # Handle combined labels like "Broad+Pulse"
    if '+' in str(label_str):
        labels = label_str.split('+')
        for label in labels:
            label = label.strip()
            if label == 'Pulse':
                label_vec[0] = 1
            elif label == 'Broad':
                label_vec[1] = 1
            elif label == 'Narrow':
                label_vec[2] = 1
    else:
        # Single label
        if label_str == 'Pulse':
            label_vec[0] = 1
        elif label_str == 'Broad':
            label_vec[1] = 1
        elif label_str == 'Narrow':
            label_vec[2] = 1

    return label_vec

# Test the function
print("Label parsing examples:")
print(f"'Pulse' -> {parse_label('Pulse')}")
print(f"'Broad+Pulse' -> {parse_label('Broad+Pulse')}")
print(f"'None' -> {parse_label('None')}")


Label parsing examples:
'Pulse' -> [1, 0, 0]
'Broad+Pulse' -> [1, 1, 0]
'None' -> [0, 0, 0]


In [17]:
def extract_advanced_features(data_segment, segment_size=256):
    """Extract comprehensive features optimized for different signal types"""
    features = {}

    # 1. Statistical features (time domain)
    mean_signal = np.mean(data_segment, axis=1)
    features['mean'] = np.mean(mean_signal)
    features['std'] = np.std(mean_signal)
    features['var'] = np.var(mean_signal)
    features['max'] = np.max(mean_signal)
    features['min'] = np.min(mean_signal)
    features['range'] = features['max'] - features['min']
    features['median'] = np.median(mean_signal)
    features['mad'] = np.median(np.abs(mean_signal - features['median']))
    features['q1'] = np.percentile(mean_signal, 25)
    features['q3'] = np.percentile(mean_signal, 75)
    features['iqr'] = features['q3'] - features['q1']
    features['skewness'] = stats.skew(mean_signal)
    features['kurtosis'] = stats.kurtosis(mean_signal)

    # 2. Frequency domain features
    fft_values = np.fft.fft(mean_signal)
    fft_abs = np.abs(fft_values)
    freqs = np.fft.fftfreq(len(mean_signal))

    # Only positive frequencies
    pos_mask = freqs > 0
    fft_abs_pos = fft_abs[pos_mask]
    freqs_pos = freqs[pos_mask]

    if len(fft_abs_pos) > 0:
        features['fft_max'] = np.max(fft_abs_pos)
        features['fft_mean'] = np.mean(fft_abs_pos)
        features['fft_std'] = np.std(fft_abs_pos)
        features['dominant_freq'] = freqs_pos[np.argmax(fft_abs_pos)]
        features['spectral_centroid'] = np.sum(freqs_pos * fft_abs_pos) / np.sum(fft_abs_pos)
        features['spectral_spread'] = np.sqrt(np.sum(((freqs_pos - features['spectral_centroid'])**2) * fft_abs_pos) / np.sum(fft_abs_pos))
    else:
        features['fft_max'] = 0
        features['fft_mean'] = 0
        features['fft_std'] = 0
        features['dominant_freq'] = 0
        features['spectral_centroid'] = 0
        features['spectral_spread'] = 0

    # 3. Wavelet features
    coeffs = pywt.wavedec(mean_signal, 'db4', level=4)
    for i, coeff in enumerate(coeffs):
        features[f'wavelet_mean_level_{i}'] = np.mean(coeff)
        features[f'wavelet_std_level_{i}'] = np.std(coeff)
        features[f'wavelet_energy_level_{i}'] = np.sum(coeff**2)

    return features

# Test the function with dummy data
print("Testing feature extraction...")
dummy_data = np.random.randn(256, 10)  # 256 time steps, 10 channels
features = extract_advanced_features(dummy_data)
print(f"Extracted {len(features)} features")
print("Sample features:", list(features.keys())[:10])


Testing feature extraction...
Extracted 34 features
Sample features: ['mean', 'std', 'var', 'max', 'min', 'range', 'median', 'mad', 'q1', 'q3']


In [18]:
# Complete the feature extraction function with all features
def extract_advanced_features_complete(data_segment, segment_size=256):
    """Extract comprehensive features optimized for different signal types"""
    features = {}

    # 1. Statistical features (time domain)
    mean_signal = np.mean(data_segment, axis=1)
    features['mean'] = np.mean(mean_signal)
    features['std'] = np.std(mean_signal)
    features['var'] = np.var(mean_signal)
    features['max'] = np.max(mean_signal)
    features['min'] = np.min(mean_signal)
    features['range'] = features['max'] - features['min']
    features['median'] = np.median(mean_signal)
    features['mad'] = np.median(np.abs(mean_signal - features['median']))
    features['q1'] = np.percentile(mean_signal, 25)
    features['q3'] = np.percentile(mean_signal, 75)
    features['iqr'] = features['q3'] - features['q1']
    features['skewness'] = stats.skew(mean_signal)
    features['kurtosis'] = stats.kurtosis(mean_signal)

    # 2. Frequency domain features
    fft_values = np.fft.fft(mean_signal)
    fft_abs = np.abs(fft_values)
    freqs = np.fft.fftfreq(len(mean_signal))

    # Only positive frequencies
    pos_mask = freqs > 0
    fft_abs_pos = fft_abs[pos_mask]
    freqs_pos = freqs[pos_mask]

    if len(fft_abs_pos) > 0:
        features['fft_max'] = np.max(fft_abs_pos)
        features['fft_mean'] = np.mean(fft_abs_pos)
        features['fft_std'] = np.std(fft_abs_pos)
        features['dominant_freq'] = freqs_pos[np.argmax(fft_abs_pos)]
        features['spectral_centroid'] = np.sum(freqs_pos * fft_abs_pos) / np.sum(fft_abs_pos)
        features['spectral_spread'] = np.sqrt(np.sum(((freqs_pos - features['spectral_centroid'])**2) * fft_abs_pos) / np.sum(fft_abs_pos))
    else:
        features['fft_max'] = 0
        features['fft_mean'] = 0
        features['fft_std'] = 0
        features['dominant_freq'] = 0
        features['spectral_centroid'] = 0
        features['spectral_spread'] = 0

    # 3. Wavelet features
    coeffs = pywt.wavedec(mean_signal, 'db4', level=4)
    for i, coeff in enumerate(coeffs):
        features[f'wavelet_mean_level_{i}'] = np.mean(coeff)
        features[f'wavelet_std_level_{i}'] = np.std(coeff)
        features[f'wavelet_energy_level_{i}'] = np.sum(coeff**2)

    # 4. Signal processing features
    # Peaks
    peaks, properties = signal.find_peaks(mean_signal, prominence=np.std(mean_signal))
    features['n_peaks'] = len(peaks)
    features['peak_mean_prominence'] = np.mean(properties['prominences']) if len(peaks) > 0 else 0
    features['peak_max_prominence'] = np.max(properties['prominences']) if len(peaks) > 0 else 0

    # Zero crossings
    zero_crossings = np.where(np.diff(np.sign(mean_signal)))[0]
    features['zero_crossing_rate'] = len(zero_crossings) / len(mean_signal)

    # Signal energy
    features['total_energy'] = np.sum(mean_signal**2)
    features['mean_energy'] = np.mean(mean_signal**2)

    # 5. 2D features (treating as image)
    features['data_mean'] = np.mean(data_segment)
    features['data_std'] = np.std(data_segment)
    features['data_contrast'] = np.max(data_segment) - np.min(data_segment)

    # Channel statistics
    channel_means = np.mean(data_segment, axis=0)
    channel_stds = np.std(data_segment, axis=0)
    features['channel_mean_mean'] = np.mean(channel_means)
    features['channel_mean_std'] = np.std(channel_means)
    features['channel_std_mean'] = np.mean(channel_stds)
    features['channel_std_std'] = np.std(channel_stds)

    # Time statistics
    time_means = np.mean(data_segment, axis=1)
    time_stds = np.std(data_segment, axis=1)
    features['time_mean_std'] = np.std(time_means)
    features['time_std_mean'] = np.mean(time_stds)

    # 6. Correlation features
    # Auto-correlation
    autocorr = np.correlate(mean_signal, mean_signal, mode='same')
    features['autocorr_max'] = np.max(autocorr)
    features['autocorr_mean'] = np.mean(autocorr)

    # Channel correlations
    if data_segment.shape[1] > 1:
        corr_matrix = np.corrcoef(data_segment.T)
        upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
        if len(upper_tri) > 0:
            features['channel_corr_mean'] = np.mean(upper_tri)
            features['channel_corr_std'] = np.std(upper_tri)
        else:
            features['channel_corr_mean'] = 0
            features['channel_corr_std'] = 0
    else:
        features['channel_corr_mean'] = 0
        features['channel_corr_std'] = 0

    # 7. Information theory features
    # Entropy
    hist, _ = np.histogram(mean_signal, bins=50)
    hist = hist[hist > 0]  # Remove zero bins
    if len(hist) > 0:
        prob = hist / np.sum(hist)
        features['entropy'] = -np.sum(prob * np.log2(prob))
    else:
        features['entropy'] = 0

    # 8. SNR estimation
    noise_estimate = np.median(np.abs(mean_signal - np.median(mean_signal))) / 0.6745
    signal_power = np.var(mean_signal)
    features['snr'] = signal_power / (noise_estimate**2) if noise_estimate > 0 else 0

    return features

# Override the previous function
extract_advanced_features = extract_advanced_features_complete

print("Complete feature extraction function loaded!")


Complete feature extraction function loaded!


In [19]:
def load_and_prepare_multilabel_data(
      data_dir='datasource/train/train/',
      limit=None,
      segment_size=256
    ):

    """Load data and extract tabular features for multi-label classification"""
    npy_files = sorted(glob.glob(os.path.join(data_dir, '*.npy')))

    if limit:
        npy_files = npy_files[:limit]

    feature_list = []
    labels = []
    file_segments = []

    for npy_file in tqdm(npy_files, desc="Processing files"):
        # Load data
        data = np.load(npy_file)
        file_base = os.path.basename(npy_file)

        # Load labels from corrected labels directory
        label_file = npy_file.replace('/train/train/', '/train-labels-corrected/train/').replace('.npy', '_labels.csv')
        if os.path.exists(label_file):
            label_df = pd.read_csv(label_file)

            # Process each segment
            for i in range(len(label_df)):
                start_idx = i * segment_size
                end_idx = start_idx + segment_size

                if end_idx <= data.shape[0]:
                    segment = data[start_idx:end_idx, :]

                    # Extract features
                    features = extract_advanced_features(segment)
                    feature_list.append(features)

                    # Convert label to multi-label format
                    label_str = label_df.iloc[i]['labels']
                    label_vec = parse_label(label_str)
                    labels.append(label_vec)

                    # Store file and segment info
                    file_segments.append({
                        'file': file_base,
                        'segment': i,
                        'label': label_str
                    })

    # Convert to DataFrame
    feature_df = pd.DataFrame(feature_list)

    return feature_df, np.array(labels), pd.DataFrame(file_segments)


In [20]:
def prepare_test_data(test_dir='datasource/test/test/',
                      segment_size=256):

    """Prepare test data for prediction"""
    npy_files = sorted(glob.glob(os.path.join(test_dir, '*.npy')))

    feature_list = []
    file_segments = []

    for npy_file in tqdm(npy_files, desc="Processing test files"):
        # Load data
        data = np.load(npy_file)
        file_base = os.path.basename(npy_file)
        file_num = int(file_base.split('.')[0])

        # Process each segment
        num_segments = data.shape[0] // segment_size
        for i in range(num_segments):
            start_idx = i * segment_size
            end_idx = start_idx + segment_size

            if end_idx <= data.shape[0]:
                segment = data[start_idx:end_idx, :]

                # Extract features
                features = extract_advanced_features(segment)
                feature_list.append(features)

                # Store file and segment info
                file_segments.append({
                    'id': f"{file_num}_{i}",
                    'file': file_base,
                    'segment': i
                })

    # Convert to DataFrame
    feature_df = pd.DataFrame(feature_list)

    return feature_df, pd.DataFrame(file_segments)


In [21]:
def train_multilabel_model(X_train, y_train, X_val, y_val):
    """Train and evaluate multilabel classification models"""

    # Initialize models - using MultiOutputClassifier for multilabel
    models = {}

    # Always include RandomForest as it's part of sklearn
    models['random_forest'] = MultiOutputClassifier(RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))

    if HAS_CATBOOST:
        models['catboost'] = MultiOutputClassifier(CatBoostClassifier(
            iterations=500,
            learning_rate=0.05,
            depth=8,
            verbose=False,
            allow_writing_files=False,
            task_type='CPU'
        ))

    if HAS_LIGHTGBM:
        models['lightgbm'] = MultiOutputClassifier(lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=64,
            feature_fraction=0.8,
            bagging_fraction=0.8,
            bagging_freq=5,
            verbose=-1
        ))

    if HAS_XGBOOST:
        models['xgboost'] = MultiOutputClassifier(xgb.XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            use_label_encoder=False,
            eval_metric='logloss'
        ))

    best_model = None
    best_score = 0

    for model_name, model in models.items():
        print(f"\\nTraining {model_name}...")

        # Train model
        model.fit(X_train, y_train)

        # Get predictions
        y_pred = model.predict(X_val)
        y_pred_proba = np.zeros((len(y_val), 3))

        # Get probabilities for each output
        for i in range(3):
            y_pred_proba[:, i] = model.estimators_[i].predict_proba(X_val)[:, 1]

        # Evaluate each signal type
        total_auc = 0
        for i, signal_type in enumerate(SIGNAL_TYPES):
            y_true_i = y_val[:, i]
            y_pred_proba_i = y_pred_proba[:, i]

            # Calculate metrics
            if len(np.unique(y_true_i)) > 1:  # Check if both classes present
                auc = roc_auc_score(y_true_i, y_pred_proba_i)
                avg_precision = average_precision_score(y_true_i, y_pred_proba_i)
                total_auc += auc

                print(f"{signal_type.capitalize()} - AUC: {auc:.4f}, AP: {avg_precision:.4f}")
                print(f"Positive samples: {np.sum(y_true_i)}/{len(y_true_i)}")
        avg_auc = total_auc / 3
        print(f"\\nAverage AUC: {avg_auc:.4f}")

        if avg_auc > best_score:
            best_score = avg_auc
            best_model = model

    return best_model


In [22]:
def generate_submission(model, test_features, test_info, scaler, threshold=0.5):

    """Generate submission file for multilabel classification"""

    # Scale features
    test_features_scaled = scaler.transform(test_features)

    # Get predictions
    y_pred_proba = np.zeros((len(test_features), 3))

    # Get probabilities for each output
    for i in range(3):
        y_pred_proba[:, i] = model.estimators_[i].predict_proba(test_features_scaled)[:, 1]

    # Print probability statistics for debugging
    print("\\nProbability statistics:")
    for i, signal_type in enumerate(['pulse', 'broad', 'narrow']):
        print(f"{signal_type}: min={y_pred_proba[:, i].min():.4f}, max={y_pred_proba[:, i].max():.4f}, mean={y_pred_proba[:, i].mean():.4f}")
        print(f"  > 0.5: {np.sum(y_pred_proba[:, i] > 0.5)}, > 0.3: {np.sum(y_pred_proba[:, i] > 0.3)}, > 0.1: {np.sum(y_pred_proba[:, i] > 0.1)}")

    # Create submission dataframe
    submission = pd.DataFrame()
    submission['id'] = test_info['id']

    # Add predictions with thresholding
    submission['pulse'] = (y_pred_proba[:, 0] > threshold).astype(int)
    submission['broad'] = (y_pred_proba[:, 1] > threshold).astype(int)
    submission['narrow'] = (y_pred_proba[:, 2] > threshold).astype(int)

    return submission


In [23]:
def generate_submission(model, test_features, test_info, scaler, threshold=0.5):
    """Generate submission file for multilabel classification"""

    # Scale features
    test_features_scaled = scaler.transform(test_features)

    # Get predictions
    y_pred_proba = np.zeros((len(test_features), 3))

    # Get probabilities for each output
    for i in range(3):
        y_pred_proba[:, i] = model.estimators_[i].predict_proba(test_features_scaled)[:, 1]

    # Print probability statistics for debugging
    print("\\nProbability statistics:")
    for i, signal_type in enumerate(['pulse', 'broad', 'narrow']):
        print(f"{signal_type}: min={y_pred_proba[:, i].min():.4f}, max={y_pred_proba[:, i].max():.4f}, mean={y_pred_proba[:, i].mean():.4f}")
        print(f"  > 0.5: {np.sum(y_pred_proba[:, i] > 0.5)}, > 0.3: {np.sum(y_pred_proba[:, i] > 0.3)}, > 0.1: {np.sum(y_pred_proba[:, i] > 0.1)}")

    # Create submission dataframe
    submission = pd.DataFrame()
    submission['id'] = test_info['id']

    # Add predictions with thresholding
    submission['pulse'] = (y_pred_proba[:, 0] > threshold).astype(int)
    submission['broad'] = (y_pred_proba[:, 1] > threshold).astype(int)
    submission['narrow'] = (y_pred_proba[:, 2] > threshold).astype(int)

    return submission


# Main Pipeline

In [24]:
# Load and prepare data
X_df, y, file_info = load_and_prepare_multilabel_data(
      data_dir='/content/train/train/',
      limit=None,
      segment_size=256
  )

print(f"\\nDataset shape: {X_df.shape}")
print(f"Number of features: {len(X_df.columns)}")

# Check if data was loaded
if len(y) == 0:
    print("ERROR: No data was loaded. Please check the data directory path.")
else:
    # Print label distribution
    print("\\nLabel distribution:")
    for i, signal_type in enumerate(SIGNAL_TYPES):
        print(f"{signal_type.capitalize()}: {np.sum(y[:, i])} positive samples ({np.sum(y[:, i])/len(y)*100:.2f}%)")

    # Check for multi-label samples
    multi_label_count = np.sum(np.sum(y, axis=1) > 1)
    print(f"\\nMulti-label samples: {multi_label_count} ({multi_label_count/len(y)*100:.2f}%)")


Processing files: 100%|██████████| 100/100 [08:08<00:00,  4.89s/it]


\nDataset shape: (96425, 55)
Number of features: 55
\nLabel distribution:
Pulse: 3953 positive samples (4.10%)
Broad: 3348 positive samples (3.47%)
Narrow: 8780 positive samples (9.11%)
\nMulti-label samples: 207 (0.21%)


In [25]:
# Handle missing/infinite values
X_df = X_df.replace([np.inf, -np.inf], np.nan)
X_df = X_df.fillna(X_df.median())

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X_df, y, test_size=0.2, random_state=42, stratify=y.sum(axis=1)
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Data preprocessing completed!")


Training set shape: (77140, 55)
Validation set shape: (19285, 55)
Data preprocessing completed!


In [26]:
# Train model
print("Training multilabel classification models...")
best_model = train_multilabel_model(X_train_scaled, y_train, X_val_scaled, y_val)

print(f"\\nBest model selected: {type(best_model).__name__}")


Training multilabel classification models...
\nTraining random_forest...
Pulse - AUC: 0.9684, AP: 0.9023
Positive samples: 797/19285
Broad - AUC: 0.9961, AP: 0.9311
Positive samples: 669/19285
Narrow - AUC: 0.9980, AP: 0.9881
Positive samples: 1751/19285
\nAverage AUC: 0.9875
\nTraining lightgbm...
Pulse - AUC: 0.9705, AP: 0.9092
Positive samples: 797/19285
Broad - AUC: 0.9961, AP: 0.9328
Positive samples: 669/19285
Narrow - AUC: 0.9984, AP: 0.9901
Positive samples: 1751/19285
\nAverage AUC: 0.9883
\nTraining xgboost...
Pulse - AUC: 0.9696, AP: 0.9098
Positive samples: 797/19285
Broad - AUC: 0.9961, AP: 0.9332
Positive samples: 669/19285
Narrow - AUC: 0.9985, AP: 0.9893
Positive samples: 1751/19285
\nAverage AUC: 0.9881
\nBest model selected: MultiOutputClassifier


In [27]:
# Generate test predictions
test_features, test_info = prepare_test_data(
    test_dir='/content/test/test/',
    segment_size=256
)

# Handle missing/infinite values in test data
test_features = test_features.replace([np.inf, -np.inf], np.nan)
test_features = test_features.fillna(test_features.median())

print(f"Test features shape: {test_features.shape}")
print(f"Test samples: {len(test_info)}")


Processing test files: 100%|██████████| 33/33 [02:19<00:00,  4.21s/it]


Test features shape: (28108, 55)
Test samples: 28108


In [30]:
# Generate submission with lower threshold
submission = generate_submission(best_model, test_features, test_info, scaler, threshold=0.3)

# Save submission
submission.to_csv('exp1_xgboost_submission.csv', index=False)
print("\\nSubmission saved to multilabel_submission.csv")
print(submission.head())

# Print summary statistics
print("\\nSubmission statistics:")
print(f"Pulse predictions: {submission['pulse'].sum()}")
print(f"Broad predictions: {submission['broad'].sum()}")
print(f"Narrow predictions: {submission['narrow'].sum()}")
print(f"Multi-label predictions: {((submission[['pulse', 'broad', 'narrow']].sum(axis=1) > 1).sum())}")

print("\\n🎉 Multi-label classification pipeline completed successfully!")


\nProbability statistics:
pulse: min=0.0000, max=1.0000, mean=0.0157
  > 0.5: 442, > 0.3: 470, > 0.1: 497
broad: min=0.0000, max=1.0000, mean=0.0131
  > 0.5: 359, > 0.3: 401, > 0.1: 463
narrow: min=0.0000, max=1.0000, mean=0.0152
  > 0.5: 422, > 0.3: 428, > 0.1: 453
\nSubmission saved to multilabel_submission.csv
    id  pulse  broad  narrow
0  0_0      0      0       0
1  0_1      0      0       0
2  0_2      0      0       0
3  0_3      0      0       0
4  0_4      0      0       0
\nSubmission statistics:
Pulse predictions: 470
Broad predictions: 401
Narrow predictions: 428
Multi-label predictions: 81
\n🎉 Multi-label classification pipeline completed successfully!


In [31]:
# Feature importance analysis
if hasattr(best_model, 'estimators_'):
    print("Feature Importance Analysis")
    print("=" * 50)

    feature_names = X_df.columns

    for i, signal_type in enumerate(SIGNAL_TYPES):
        print(f"\\n{signal_type.capitalize()} Signal Classification:")
        if hasattr(best_model.estimators_[i], 'feature_importances_'):
            importances = best_model.estimators_[i].feature_importances_
            indices = np.argsort(importances)[::-1]

            print("Top 10 most important features:")
            for j in range(min(10, len(feature_names))):
                print(f"{j+1:2d}. {feature_names[indices[j]]:30s} ({importances[indices[j]]:.4f})")
        else:
            print("Feature importance not available for this model type.")

print("\\n" + "="*50)
print("Analysis completed! 📊")


Feature Importance Analysis
\nPulse Signal Classification:
Top 10 most important features:
 1. channel_corr_std               (1389.0000)
 2. channel_mean_std               (1188.0000)
 3. channel_std_std                (1168.0000)
 4. data_contrast                  (1155.0000)
 5. channel_std_mean               (1071.0000)
 6. spectral_spread                (957.0000)
 7. wavelet_mean_level_2           (946.0000)
 8. wavelet_mean_level_1           (923.0000)
 9. wavelet_mean_level_4           (884.0000)
10. wavelet_mean_level_3           (872.0000)
\nBroad Signal Classification:
Top 10 most important features:
 1. wavelet_mean_level_3           (1238.0000)
 2. wavelet_mean_level_1           (1162.0000)
 3. wavelet_mean_level_4           (1151.0000)
 4. data_contrast                  (1139.0000)
 5. wavelet_mean_level_2           (1128.0000)
 6. channel_corr_std               (1083.0000)
 7. channel_mean_std               (1073.0000)
 8. peak_max_prominence            (966.0000)
 9. sn